# 11.01 · Tracing & observability

**Chapter:** LLMOps, Observability & Safety · **Milestone:** M5

## Goals
- [ ] Traces, spans and attributes; the OpenTelemetry (OTel) GenAI conventions and OpenInference
- [ ] Auto-instrument every LLM call, and add your own spans for retrieval, tools and business steps
- [ ] Run Arize Phoenix locally and browse traces; know Langfuse, LangSmith, W&B Weave, Logfire, Helicone

## Resources
- [Arize Phoenix docs](https://arize.com/docs/phoenix) · [OpenInference](https://github.com/Arize-ai/openinference) — free/OSS
- [Langfuse docs](https://langfuse.com/docs) — free/OSS (self-host) + free cloud tier
- [OpenTelemetry – GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) — free

| Tool | Hosting | Notes |
|---|---|---|
| **Arize Phoenix** | runs locally in-process / self-host / cloud | OTel-native, evals + datasets built in; used here (no Docker needed) |
| **Langfuse** | self-host (Docker) / cloud free tier | OSS (MIT), prompts + evals + datasets + cost; very popular |
| **LangSmith** | cloud (free tier) / enterprise self-host | best with LangChain/LangGraph; accepts OTel |
| **W&B Weave**, **Logfire** (Pydantic), **Helicone**, **Braintrust**, **Opik** | mostly cloud w/ free tiers | similar core; differ in evals/UX |

A **trace** is one request end to end; it's a tree of **spans** (LLM call, retrieval, tool call), each with timing and attributes
(model, tokens, prompt, output, cost). Without traces, debugging an agent is guessing.

In [ ]:
import os
import time

from aieng import DATA, rag

os.environ["PHOENIX_WORKING_DIR"] = str(DATA / "phoenix")  # keep Phoenix's SQLite DB inside data/
import phoenix as px

session = px.launch_app()  # local server + UI at http://localhost:6006
print("Phoenix UI:", session.url)

## 1. Wire OTel: Phoenix exporter + an in-memory copy for this notebook
`phoenix.otel.register` creates an OTel `TracerProvider` that ships spans to Phoenix. We add an in-memory exporter too, so we can print spans here.
Then **OpenInference auto-instrumentation** patches the `openai` SDK: every chat call becomes a span with prompts, outputs and token counts.

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from phoenix.otel import register

tracer_provider = register(project_name="ai-eng", endpoint="http://localhost:6006/v1/traces", batch=False, verbose=False)
memory = InMemorySpanExporter()
tracer_provider.add_span_processor(SimpleSpanProcessor(memory))
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer("ai-eng")

## 2. Trace the RAG pipeline with your own spans
Auto-instrumentation sees the LLM call; it can't see retrieval. Wrap each step in a span and attach what you'd want when debugging.
OpenInference span kinds (`CHAIN`, `RETRIEVER`, `LLM`, `TOOL`, `AGENT`) make the UI render them properly.

In [ ]:
from opentelemetry.trace import Status, StatusCode

from aieng.llm import chat


def traced_answer(question, k=5, user_id="anonymous"):
    with tracer.start_as_current_span("rag_answer", attributes={"openinference.span.kind": "CHAIN", "input.value": question, "user.id": user_id}) as root:
        with tracer.start_as_current_span("retrieve", attributes={"openinference.span.kind": "RETRIEVER", "input.value": question}) as span:
            hits = rag.search(question, k)
            for i, (pid, score, text) in enumerate(hits):
                span.set_attribute(f"retrieval.documents.{i}.document.id", str(pid))
                span.set_attribute(f"retrieval.documents.{i}.document.score", score)
                span.set_attribute(f"retrieval.documents.{i}.document.content", text[:500])
        sources = "\n\n".join(f"[{pid}] {text}" for pid, _, text in hits)
        try:
            answer = chat(rag.PROMPT.format(sources=sources, question=question), temperature=0, max_tokens=300)  # auto-instrumented LLM span
        except Exception as e:
            root.set_status(Status(StatusCode.ERROR, str(e))); raise
        root.set_attribute("output.value", answer)
        return answer


for q in ["When was Abraham Lincoln born?", "What is the capital of Uruguay?", "Who won the 2022 World Cup?"]:
    print(q, "->", traced_answer(q, user_id="ana")[:90])

## 3. Inspect the spans
Open the Phoenix UI (link above) → project **ai-eng** to see the trace tree. Here, the same spans from the in-memory copy:

In [ ]:
spans = memory.get_finished_spans()
for s in spans[-6:]:
    a = dict(s.attributes)
    dur = (s.end_time - s.start_time) / 1e6
    tokens = a.get("llm.token_count.total")
    print(f"{s.name:24} kind={a.get('openinference.span.kind', '?'):9} {dur:7.0f} ms"
          + (f" | model={a.get('llm.model_name')} tokens={tokens}" if tokens else "")
          + f" | trace={format(s.context.trace_id, '032x')[:8]}")

What you get from this for free: latency per step (is retrieval or the LLM slow?), token counts and cost per request, the exact prompt the
model saw (critical when frameworks build prompts for you), errors with context, and filters by user/feature/model.

## 4. Agents and frameworks
The same instrumentation captures tool-calling loops (each LLM turn is a span). Frameworks have their own instrumentors:
`openinference-instrumentation-langchain`, `-llama-index`, `-openai-agents`, `-dspy`, `-smolagents`, `-litellm`… One line each.

In [ ]:
import json

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
TOOLS = [{"type": "function", "function": {"name": "search_docs", "description": "Search the encyclopedia.",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]

with tracer.start_as_current_span("agent_run", attributes={"openinference.span.kind": "AGENT", "input.value": "Lincoln's wife"}):
    messages = [{"role": "user", "content": "Who was Lincoln's wife? Use search_docs."}]
    for step in range(3):
        msg = c.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0, max_tokens=400, **EXTRA).choices[0].message
        if not msg.tool_calls:
            break
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            with tracer.start_as_current_span("search_docs", attributes={"openinference.span.kind": "TOOL", "input.value": tc.function.arguments}):
                out = "\n".join(t[:300] for _, _, t in rag.search(json.loads(tc.function.arguments)["query"], 3))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
print(msg.content)
print([s.name for s in memory.get_finished_spans()[-5:]])

## 5. Langfuse (cloud free tier or self-hosted)
Drop-in: swap the import and every call is traced; `@observe` creates spans for your functions. Runs if `LANGFUSE_PUBLIC_KEY` /
`LANGFUSE_SECRET_KEY` / `LANGFUSE_HOST` are in `.env` (free account at cloud.langfuse.com, or self-host with Docker Compose).

In [ ]:
if os.getenv("LANGFUSE_PUBLIC_KEY"):
    from langfuse import observe
    from langfuse.openai import OpenAI as LangfuseOpenAI

    lf = LangfuseOpenAI(base_url=str(c.base_url), api_key=c.api_key)

    @observe()
    def lf_answer(question):
        hits = rag.search(question, 3)
        return lf.chat.completions.create(model=MODEL, **EXTRA, messages=[{"role": "user", "content": rag.PROMPT.format(
            sources="\n".join(f"[{p}] {t}" for p, _, t in hits), question=question)}]).choices[0].message.content

    print(lf_answer("When was Lincoln born?"))
else:
    print("LANGFUSE_* keys not set — skipping (free at cloud.langfuse.com)")

## 6. What to log in production
- **Always**: model + version, prompt version (06.03), token counts, latency, cost, errors, user/session/feature ids.
- **Usually**: full prompts and outputs — mind **PII** (redact, 11.04), retention and access controls.
- **Sample** when volume is huge, but keep 100% of errors and negative feedback (11.05).

In [ ]:
OpenAIInstrumentor().uninstrument()
px.close_app()

## Exercises
1. Instrument the LangChain or LlamaIndex pipeline from 08.03 with its OpenInference instrumentor (`uv add openinference-instrumentation-langchain`).
   Find the hidden framework prompt in the trace.
2. Add a `cost_usd` attribute to each LLM span (05.05 prices) and compute total cost per trace from the spans.
3. Create a free Langfuse cloud project and send the same traces there. Compare the UIs.

In [ ]:
# your code here